# Sales Analytics (Standalone)

Input: `csv/Brigade_Bangalore_10_April_26 (1)bc6219c.csv`

## 1. Load Data

In [3]:
from pathlib import Path

import json
import pandas as pd

from pathlib import Path

ROOT_DIR = Path.cwd().parents[1]

INPUT_CSV = ROOT_DIR / "csv" / "Brigade_Bangalore_10_April_26 (1)bc6219c.csv"
OUTPUT_DIR = ROOT_DIR / "outputs"
SALES_SUMMARY_PATH = OUTPUT_DIR / "sales_summary.json"
BRAND_SUMMARY_PATH = OUTPUT_DIR / "brand_summary.json"
CATEGORY_SUMMARY_PATH = OUTPUT_DIR / "category_summary.json"
SALESPERSON_SUMMARY_PATH = OUTPUT_DIR / "salesperson_summary.json"

NUMERIC_COLUMNS = ["qty", "GMV", "NMV"]

def coerce_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").fillna(0)

def normalize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for column in NUMERIC_COLUMNS:
        if column in df.columns:
            df[column] = coerce_numeric(df[column])
    if "brand_name" in df.columns:
        df["brand_name"] = df["brand_name"].fillna("Unknown")
    if "dep_name" in df.columns:
        df["dep_name"] = df["dep_name"].fillna("Unknown")
    if "salesperson_name" in df.columns:
        df["salesperson_name"] = df["salesperson_name"].fillna("Unknown")
    return df

def resolve_order_count(df: pd.DataFrame) -> int:
    for column in ["order_id", "invoice_number"]:
        if column in df.columns:
            return int(df[column].dropna().nunique())
    return int(len(df))

def round_value(value: float, decimals: int = 2) -> float:
    return float(round(value, decimals))

def build_sales_summary(df: pd.DataFrame) -> dict:
    total_orders = resolve_order_count(df)
    total_qty = df["qty"].sum() if "qty" in df.columns else 0.0
    total_gmv = df["GMV"].sum() if "GMV" in df.columns else 0.0
    total_nmv = df["NMV"].sum() if "NMV" in df.columns else 0.0
    average_bill = total_nmv / total_orders if total_orders else 0.0
    return {
        "total_orders": int(total_orders),
        "total_qty": round_value(total_qty),
        "total_gmv": round_value(total_gmv),
        "total_nmv": round_value(total_nmv),
        "average_bill": round_value(average_bill)
    }

def build_group_summary(df: pd.DataFrame, group_column: str, output_key: str) -> pd.DataFrame:
    grouped = (
        df.groupby(group_column, dropna=False)
        .agg(total_qty=("qty", "sum"), total_nmv=("NMV", "sum"))
        .reset_index()
    )
    grouped["total_qty"] = grouped["total_qty"].fillna(0)
    grouped["total_nmv"] = grouped["total_nmv"].fillna(0)
    grouped = grouped.sort_values(by=["total_nmv", group_column], ascending=[False, True])
    grouped["ranking"] = range(1, len(grouped) + 1)
    grouped = grouped.rename(columns={group_column: output_key})
    grouped["total_qty"] = grouped["total_qty"].apply(round_value)
    grouped["total_nmv"] = grouped["total_nmv"].apply(round_value)
    return grouped[[output_key, "total_qty", "total_nmv", "ranking"]]

def write_json(path: Path, payload) -> None:
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"Saved {path.as_posix()}")

df = pd.read_csv(INPUT_CSV)
df = normalize_dataframe(df)
df.head()

,order_id,coupon_code,offer_name,discount_code,invoice_number,invoice_type,order_date,order_time,return_id,store_id,...,NMV,coupon_amount,item_promotion,amt_without_gwp,total_amount,pb_eb_sale,week_assigned,tax_m,taxable_amt,tax_amt
0,104363838,NaN,Buy 2 Get 1 on PB,NaN,ML0426KAP0001358,sales,10-04-2026,16:55:36,NaN,ST1008,...,274.36,0.00,125.64,274.36,274.36,274.36,NaN,1.18,232.51,41.85
1,104377545,NaN,Buy 2 Get 2 on Sheet Mask,NaN,ML0426KAP0001399,sales,10-04-2026,19:21:55,NaN,ST1008,...,99.00,0.00,99.00,99.00,99.00,99.00,NaN,1.18,83.90,15.10
2,104362899,NaN,Buy 2 Get 1 Faces and Ny bae,NaN,ML0426KAP0001353,sales,10-04-2026,16:45:32,NaN,ST1008,...,553.17,0.00,245.83,553.17,553.17,553.17,NaN,1.18,468.79,84.38
3,104373042,MAR2620,NaN,MAR2620,ML0426KAP0001384,sales,10-04-2026,18:41:51,NaN,ST1008,...,1799.00,350.82,0.00,1448.18,1448.18,1799.00,NaN,1.18,1227.27,220.91
4,104375288,NaN,Buy 2 Get 1 Faces and Ny bae,NaN,ML0426KAP0001393,sales,10-04-2026,19:02:09,NaN,ST1008,...,466.67,0.00,232.33,466.67,466.67,466.67,NaN,1.18,395.48,71.19


## 2. Dataset Overview

In [4]:
pd.DataFrame({"rows": [len(df)], "columns": [len(df.columns)]})

,rows,columns
0,101,39


In [5]:
df.describe(include="all").T.head(12)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
order_id,101.0,NaN,NaN,NaN,104361978.811881,15872.97153,104338647.0,104341290.0,104363838.0,104377545.0,104391745.0
coupon_code,3,1,MAR2620,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
offer_name,73,9,Buy 2 Get 1 Faces and Ny bae,35,NaN,NaN,NaN,NaN,NaN,NaN,NaN
discount_code,3,1,MAR2620,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
invoice_number,101,24,ML0426KAP0001324,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
invoice_type,101,1,sales,101,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_date,101,1,10-04-2026,101,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_time,101,24,12:42:18,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
return_id,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
store_id,101,1,ST1008,101,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Sales Summary

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sales_summary = build_sales_summary(df)
write_json(SALES_SUMMARY_PATH, sales_summary)
pd.DataFrame([sales_summary])

Saved c:/Users/vipee/Desktop/study/yolo/outputs/sales_summary.json


,total_orders,total_qty,total_gmv,total_nmv,average_bill
0,24,117.0,44920.0,34831.74,1451.32


## 4. Brand Analysis

In [7]:
brand_summary = build_group_summary(df, "brand_name", "brand_name")
write_json(BRAND_SUMMARY_PATH, brand_summary.to_dict(orient="records"))
brand_summary.head(10)

Saved c:/Users/vipee/Desktop/study/yolo/outputs/brand_summary.json


,brand_name,total_qty,total_nmv,ranking
7,Faces Canada,33.0,15697.21,1
16,NY Bae,10.0,2342.60,2
3,COSRX,2.0,2070.00,3
15,Maybelline,3.0,1834.29,4
20,Round Lab,1.0,1799.00,5
6,DERMDOC,6.0,1620.90,6
11,Good Vibes,29.0,1485.00,7
2,Beauty of Joseon,1.0,1278.00,8
1,Bare Anatomy,2.0,1191.60,9
12,Juicy Chemistry,2.0,750.00,10


## 5. Category Analysis

In [8]:
category_summary = build_group_summary(df, "dep_name", "dep_name")
write_json(CATEGORY_SUMMARY_PATH, category_summary.to_dict(orient="records"))
category_summary.head(10)

Saved c:/Users/vipee/Desktop/study/yolo/outputs/category_summary.json


,dep_name,total_qty,total_nmv,ranking
3,makeup,55.0,21939.09,1
5,skin,42.0,9408.28,2
2,hair,6.0,1957.15,3
4,personal-care,4.0,763.80,4
0,bath-and-body,9.0,514.42,5
1,fragrance,1.0,249.00,6


## 6. Salesperson Analysis

In [9]:
salesperson_summary = build_group_summary(df, "salesperson_name", "salesperson_name")
write_json(SALESPERSON_SUMMARY_PATH, salesperson_summary.to_dict(orient="records"))
salesperson_summary.head(10)

Saved c:/Users/vipee/Desktop/study/yolo/outputs/salesperson_summary.json


,salesperson_name,total_qty,total_nmv,ranking
4,Zufishan Khazra,53.0,16583.38,1
5,kasthuri v,20.0,6541.95,2
2,Shashikala .,12.0,4631.70,3
1,Priya v,16.0,3665.75,4
0,Naziya Begum,9.0,3408.96,5
3,Unknown,7.0,0.00,6


## 7. Top 10 Brands

In [10]:
brand_summary.head(10)

,brand_name,total_qty,total_nmv,ranking
7,Faces Canada,33.0,15697.21,1
16,NY Bae,10.0,2342.60,2
3,COSRX,2.0,2070.00,3
15,Maybelline,3.0,1834.29,4
20,Round Lab,1.0,1799.00,5
6,DERMDOC,6.0,1620.90,6
11,Good Vibes,29.0,1485.00,7
2,Beauty of Joseon,1.0,1278.00,8
1,Bare Anatomy,2.0,1191.60,9
12,Juicy Chemistry,2.0,750.00,10


## 8. Top 10 Categories

In [11]:
category_summary.head(10)

,dep_name,total_qty,total_nmv,ranking
3,makeup,55.0,21939.09,1
5,skin,42.0,9408.28,2
2,hair,6.0,1957.15,3
4,personal-care,4.0,763.80,4
0,bath-and-body,9.0,514.42,5
1,fragrance,1.0,249.00,6


## 9. Top 10 Salespersons

In [12]:
salesperson_summary.head(10)

,salesperson_name,total_qty,total_nmv,ranking
4,Zufishan Khazra,53.0,16583.38,1
5,kasthuri v,20.0,6541.95,2
2,Shashikala .,12.0,4631.70,3
1,Priya v,16.0,3665.75,4
0,Naziya Begum,9.0,3408.96,5
3,Unknown,7.0,0.00,6
